# 02 — Kafka/MSK streaming ingest (EMR)

Read clickstream events from a live Amazon MSK topic with Spark Structured Streaming and land them as an append-only bronze Delta table on S3.

**Requires:**
1. An MSK cluster reachable from the EMR cluster's VPC/subnets (see `infra/terraform/`).
2. A topic (default `retail-clickstream`) — created automatically by the cell below via `retail_lakehouse.kafka_admin.create_topics` (installed on this node as the wheel's `[kafka]` extra); idempotent, so re-running this notebook against an existing topic is a no-op.
3. Producer traffic on that topic — seeded automatically by the "Seed producer traffic" cell below (synthetic events, produced directly from this cluster); skip that cell if you already have real producer traffic on `kafka_topic`, e.g. from `scripts/seed_kafka_topic.py` run over an SSM port-forward tunnel to the broker.
4. The `aws-msk-iam-auth` and `spark-sql-kafka-0-10` packages on the classpath -- the `%%configure` cell below adds both via `spark.jars.packages` (this is a genuine setup step neither EMR nor this notebook automates any other way).

If MSK isn't provisioned yet, this notebook raises a clear error rather than silently doing nothing — use `01_batch_lakehouse_bronze_silver_gold.ipynb` or `07_file_rate_streaming_fallback.ipynb` to keep working while infra is being set up.

Run the cell below first, before anything else -- it configures Delta Lake for this notebook's Spark session (EMR doesn't bundle Delta by default, unlike Databricks). `%%configure -f` must run before any other Spark code in this session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,software.amazon.msk:aws-msk-iam-auth:2.2.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "retail_lakehouse"
base_path = "s3://<your-lakehouse-bucket>/data"
kafka_bootstrap_servers = ""  # e.g. b-1.xxx.kafka.us-east-1.amazonaws.com:9098 -- from `terraform output -raw msk_bootstrap_brokers_command`
kafka_topic = "retail-clickstream"
starting_offsets = "earliest"

from retail_lakehouse.config import PipelineConfig
cfg = PipelineConfig(schema=schema, base_path=base_path,
                      kafka_bootstrap_servers=kafka_bootstrap_servers, kafka_topic=kafka_topic)
spark.sql(f"USE `{cfg.schema}`")

if not kafka_bootstrap_servers:
    raise ValueError("Set kafka_bootstrap_servers before running this notebook.")

# Idempotent -- safe to re-run. Removes the need for a separate SSM session or CLI setup just to
# create the topic before running this notebook. Passes kafka_topic explicitly so the topic actually
# created always matches the one subscribed below, even if kafka_topic is changed from its default.
from retail_lakehouse.kafka_admin import create_topics, TopicSpec
create_topics(kafka_bootstrap_servers, topics=[TopicSpec(name=kafka_topic)])

## Seed producer traffic

`scripts/seed_kafka_topic.py` can reach MSK Serverless from a laptop via SSM port forwarding to the
broker port on an in-VPC node (same pattern as the JupyterHub tunnel in step 5 of `RUNBOOK_AWS.md`), but
that means starting a separate tunnel session first. The cell below instead publishes the same seed
events `00_environment_setup.ipynb` already generated (`cfg.path("source", "events_json")`) directly
from this cluster, so the streaming path here processes the same data as the batch path in `01` for a
fair comparison, without a separate tunnel session. Safe to skip if you already have real producer
traffic on `kafka_topic`; safe to re-run otherwise, it just republishes the same seed events again.

In [ ]:
from retail_lakehouse.generate import produce_to_kafka
from retail_lakehouse.kafka_admin import _MSKTokenProvider

# Reuse the same seed events `00_environment_setup.ipynb` generated (JSON at cfg.path("source",
# "events_json")), so the streaming path processes the same data as the batch path in `01`, instead of
# an independently generated random set. Requires notebook 00 to have been run at least once already.
event_rows = [row.asDict() for row in spark.read.json(cfg.path("source", "events_json")).collect()]

sent = produce_to_kafka(
    bootstrap_servers=kafka_bootstrap_servers,
    topic=kafka_topic,
    rows=event_rows,
    security_protocol="SASL_SSL",
    sasl_mechanism="OAUTHBEARER",
    sasl_oauth_token_provider=_MSKTokenProvider("us-east-1"),
)
print(f"Produced {sent} seed events (from notebook 00) to topic '{kafka_topic}'")

## Configure the Kafka source

IAM auth (`AWS_MSK_IAM`) uses the EMR instance profile's credentials ambiently -- no service-credential indirection needed, unlike the Databricks-serverless version of this notebook.

In [ ]:
from pyspark.sql import functions as F

reader = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", kafka_topic)
    .option("startingOffsets", starting_offsets)
    .option("failOnDataLoss", "false")
    .option("maxOffsetsPerTrigger", 10000)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "AWS_MSK_IAM")
    .option("kafka.sasl.jaas.config", "software.amazon.msk.auth.iam.IAMLoginModule required;")
    .option("kafka.sasl.client.callback.handler.class", "software.amazon.msk.auth.iam.IAMClientCallbackHandler")
)

raw_kafka = reader.load()

## Parse and write to bronze

`parse_kafka_value` (from `retail_lakehouse.transformations`) is unit-tested in `tests/test_kafka_parsing.py` against a synthetic Kafka-shaped DataFrame, so this logic is verified in CI without needing a live broker.

In [ ]:
from retail_lakehouse.transformations import parse_kafka_value, add_ingest_metadata

parsed = parse_kafka_value(raw_kafka)
bronze_stream = add_ingest_metadata(parsed, "kafka_msk")

query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", cfg.checkpoint("kafka_bronze_clickstream"))
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(cfg.table("bronze_clickstream_kafka"))
)
query.awaitTermination()

print("Micro-batch(es) complete. Query progress:")
for p in query.recentProgress[-5:]:
    print({"timestamp": p["timestamp"], "numInputRows": p["numInputRows"], "durationMs": p["durationMs"]})

spark.table(cfg.table("bronze_clickstream_kafka")).orderBy(F.desc("_ingest_ts")).limit(20).show(truncate=False)

## Next

`03_streaming_silver_gold_delta.ipynb` reads `bronze_clickstream_kafka` as a stream and builds the silver/gold layers with watermarking, de-duplication, and an idempotent upsert into gold.